In [1]:
import html, json
import pandas as pd

from IPython.display import HTML, display


# sample dataset
data = "data/gsv3/grooveseeker_radio/gsv3_radio_entities_altcountry_sample.json"

# full dataset - buy access - verdantintel.com
# data = "../verdant_intelligence/exports/grooveseeker_entities/grooveseeker_radio_entities_altcountry.json"

In [2]:
song_df = pd.read_json(data)
song_df = song_df.sample(frac=1).reset_index(drop=True)

song_df.head()

,entity,song,spotify_url,popularity,album,image
0,Kumo 99,Tiny Twist,https://open.spotify.com/track/3xweC8YgNjaRRjU...,30,Body N. Will,https://i.scdn.co/image/ab67616d0000b2739409bf...
1,The Grey Owls,Like a Sparrow,https://open.spotify.com/track/0luRLVBcQf59CCL...,0,Murderer of Love,https://i.scdn.co/image/ab67616d0000b2732b1e1a...
2,The Felice Brothers,Inferno,https://open.spotify.com/track/6H66K7PGQMdXMlI...,37,From Dreams to Dust,https://i.scdn.co/image/ab67616d0000b273db955c...
3,Dutch Vanderpool,Deuteronomy 4: 39,https://open.spotify.com/track/6YeFNOOdqwjkm0u...,0,The Word in My Heart,https://i.scdn.co/image/ab67616d0000b27374847c...
4,The Head and the Heart,Let's Be Still,https://open.spotify.com/track/3QmzlL0tTtDgD1H...,62,Let's Be Still,https://i.scdn.co/image/ab67616d0000b2731b5e06...


In [3]:
song_df.shape

(100, 6)

I have given you a sample dataset of 100 songs to explore. 

# Play Music

In [4]:
def show_radio(song_df):
    tracks = []

    for _, row in song_df.iterrows():
        tracks.append(
            {
                "song": row["song"],
                "artist": row["entity"],
                "url": row["spotify_url"],
            }
        )

    tracks_json = json.dumps(tracks)

    radio_html = """
    <!DOCTYPE html>
    <html>
    <body>

    <div id="radio-label" style="margin-bottom:10px;"></div>
    <div id="spotify-player"></div>

    <div style="margin-top:10px;">
        <button id="back">← Back</button>
        <button id="forward">Forward →</button>
    </div>

    <script src="https://open.spotify.com/embed/iframe-api/v1" async></script>

    <script>
    const tracks = %s;

    let position = 0;
    let controller = null;
    let advancing = false;

    function updateLabel() {
        const row = tracks[position];

        document.getElementById("radio-label").innerHTML =
            "<b>" + row.song + "</b><br>" +
            row.artist + "<br>" +
            "Song " + (position + 1) + " of " + tracks.length;
    }

    function loadSong(step, autoplay=false) {
        position = (position + step + tracks.length) %% tracks.length;

        updateLabel();
        advancing = false;

        controller.loadEntity(tracks[position].url);

        if (autoplay) {
            setTimeout(() => controller.play(), 500);
        }
    }

    window.onSpotifyIframeApiReady = (IFrameAPI) => {
        const element = document.getElementById("spotify-player");

        IFrameAPI.createController(
            element,
            {
                url: tracks[position].url,
                width: "100%%",
                height: 152
            },
            (EmbedController) => {
                controller = EmbedController;

                controller.addListener("playback_update", (event) => {
                    const state = event.data;

                    if (
                        state.duration > 0 &&
                        state.position >= state.duration - 750 &&
                        !advancing
                    ) {
                        advancing = true;
                        loadSong(1, true);
                    }
                });
            }
        );
    };

    document.getElementById("back").onclick = () => loadSong(-1);
    document.getElementById("forward").onclick = () => loadSong(1);

    updateLabel();
    </script>

    </body>
    </html>
    """ % tracks_json

    display(
        HTML(
            '<iframe srcdoc="{}" width="100%" height="260" '
            'style="border:0;"></iframe>'.format(
                html.escape(radio_html, quote=True)
            )
        )
    )

In [6]:
song_df = song_df.sample(frac=1).reset_index(drop=True)

show_radio(song_df)